
# Tech Challenge — Fase 4
## Preparação da Base de Conhecimento para RAG

### 1. Objetivo

Preparar os comentários de clientes do conjunto de dados
Brazilian E-Commerce Public Dataset by Olist para compor
a base de conhecimento de um sistema de Retrieval-Augmented
Generation (RAG).

### 2. Contexto

A análise exploratória realizada no notebook
`01_exploracao_dados.ipynb` identificou 99.224 registros
de avaliações, dos quais 42.687 apresentam conteúdo
textual efetivo no título, na mensagem ou em ambos.

Esses registros constituem o conjunto inicial de
candidatos à base de conhecimento.

### 3. Escopo desta etapa

- Selecionar avaliações com conteúdo textual efetivo.
- Definir uma representação textual para cada avaliação.
- Aplicar procedimentos de limpeza que preservem o significado dos comentários.
- Definir os identificadores e metadados relevantes.
- Preparar documentos adequados à futura geração de embeddings.
- Validar a integridade e a rastreabilidade dos documentos preparados.

### 4. Relação com o Tech Challenge

Esta etapa contribui para a construção da base de
conhecimento utilizada pelo sistema RAG.

A geração de embeddings, o armazenamento vetorial,
a recuperação semântica e a geração de respostas
fundamentadas serão desenvolvidos nas etapas seguintes.

### 5. Princípios metodológicos

- Preservar os arquivos originais da Olist.
- Evitar exclusões sem justificativa técnica.
- Manter a associação entre os textos e seus registros de origem.
- Documentar as transformações realizadas.
- Priorizar os requisitos do enunciado da Fase 4.
- Garantir que o processamento possa ser reproduzido
  independentemente do notebook de exploração.


## Execução no Databricks

Preservamos a sequência didática de **Objetivo, Justificativa e Ação** e as análises originais.
O código fica no Git folder; os dados ficam no Volume indicado em `data_root`.

**Diretório padrão da Fase 4:** `/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase4`.
**Localização física no S3:** `s3://s3tc2/projetos/fiap/tech_challenge_fase4/`.
Reutilizamos o Volume `workspace.default.vol_trio_drive` documentado no setup da Fase 3.
A configuração compartilhada já fornece esse caminho; o Volume deve estar acessível ao compute.
Se houver um widget `data_root` preenchido com outro endereço, ajuste-o para o caminho acima.

As análises numéricas em Markdown são **resultados de referência da execução local anterior**.
As saídas das células foram limpas; os resultados da nova execução devem ser conferidos antes
de serem apresentados como resultados Databricks. A preparação não altera os arquivos da camada raw.


In [0]:
# Objetivo: carregar a configuração compartilhada do Databricks.
# Justificativa: separar o código versionado no Git dos dados no Volume/S3.
# Ação: localizar src, importar bibliotecas e obter os parâmetros do notebook.
import sys
from pathlib import Path
import pandas as pd
import numpy as np

DIRETORIO_ATUAL = Path.cwd().resolve()
# Módulos de apoio no mesmo Volume da Fase 4 (independente do Git folder).
DIRETORIO_CODIGO_VOLUME = Path(
    "/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase4/src"
)
ARQUIVOS_MODULOS = ["__init__.py", "config.py", "artefatos.py", "busca_vetorial.py"]
modulos_ausentes = [nome for nome in ARQUIVOS_MODULOS
                   if not (DIRETORIO_CODIGO_VOLUME / nome).is_file()]
if modulos_ausentes:
    raise FileNotFoundError(
        f"Envie os arquivos da pasta src do pacote para {DIRETORIO_CODIGO_VOLUME}. "
        f"Arquivos ausentes: {modulos_ausentes}"
    )
# Importar módulos Python a partir de cópia local na sessão do compute.
# Os originais no Volume/S3 são preservados.
import shutil
import tempfile
RAIZ_CODIGO = Path(tempfile.mkdtemp(prefix="olist_fase4_codigo_"))
(RAIZ_CODIGO / "src").mkdir()
for nome in ARQUIVOS_MODULOS:
    shutil.copyfile(DIRETORIO_CODIGO_VOLUME / nome, RAIZ_CODIGO / "src" / nome)
# Evitar reutilizar módulos de outra execução ou de outro projeto na mesma sessão.
for nome_modulo in list(sys.modules):
    if nome_modulo == "src" or nome_modulo.startswith("src."):
        del sys.modules[nome_modulo]
if str(RAIZ_CODIGO) not in sys.path:
    sys.path.insert(0, str(RAIZ_CODIGO))
from src.config import configurar, ARQUIVOS_RAW, NOME_MODELO, REVISAO_MODELO
from src.artefatos import ambiente, validar_documentos, carregar_documentos
CFG = configurar(dbutils)
BASE_DIR = CFG["base"]
REVIEWS_PATH = CFG["raw"] / "olist_order_reviews_dataset.csv"
ARQUIVO_DOCUMENTOS = CFG["documentos"]
ARQUIVO_EMBEDDINGS = CFG["embeddings"]

if not REVIEWS_PATH.is_file():
    raise FileNotFoundError(f"CSV não encontrado: {REVIEWS_PATH}.")


In [0]:

# Objetivo:
#
# Carregar a tabela original de avaliações da Olist
# e validar sua estrutura inicial.
#
# Justificativa:
#
# Garantir que a preparação da base de conhecimento
# utilize os mesmos dados investigados anteriormente,
# mantendo a rastreabilidade e a reprodutibilidade.
#
# Ação:
#
# Ler o arquivo CSV, verificar suas dimensões,
# identificar as colunas e visualizar uma amostra,
# sem modificar os registros originais.

# Carregamento da tabela de avaliações
df_reviews = pd.read_csv(REVIEWS_PATH, dtype={"review_id": str, "order_id": str})

# Verificação das dimensões
print("Quantidade de registros:", df_reviews.shape[0])
print("Quantidade de colunas:", df_reviews.shape[1])

# Identificação das colunas
print("\nColunas disponíveis:")
print(df_reviews.columns.tolist())

# Visualização dos primeiros registros
print("\nAmostra dos dados:")
display(df_reviews.head())


In [0]:

# Objetivo:
#
# Selecionar as avaliações que possuem conteúdo
# textual efetivo no título, na mensagem ou em ambos.
#
# Justificativa:
#
# A base de conhecimento do RAG será construída
# a partir dos comentários dos clientes.
# Registros sem conteúdo textual não fornecem
# texto para a geração de embeddings.
#
# Ação:
#
# Identificar títulos e mensagens não vazios,
# criar um novo DataFrame com os registros elegíveis
# e validar a quantidade selecionada,
# preservando o DataFrame original.

# Identificação de títulos com conteúdo efetivo
titulo_valido = (
    df_reviews["review_comment_title"]
    .fillna("")
    .str.strip()
    .ne("")
)

# Identificação de mensagens com conteúdo efetivo
mensagem_valida = (
    df_reviews["review_comment_message"]
    .fillna("")
    .str.strip()
    .ne("")
)

# Critério de seleção: título OU mensagem válida
possui_texto = titulo_valido | mensagem_valida

# Criação do DataFrame de trabalho
df_reviews_texto = df_reviews.loc[
    possui_texto
].copy()

# Validação da seleção
print(
    "Total de avaliações originais:",
    len(df_reviews)
)

print(
    "Avaliações com texto efetivo:",
    len(df_reviews_texto)
)

print(
    "Avaliações sem texto efetivo:",
    (~possui_texto).sum()
)

print(
    "Percentual selecionado:",
    round(possui_texto.mean() * 100, 2),
    "%"
)

print(
    "Total original preservado:",
    len(df_reviews)
)


In [0]:

# Objetivo:
#
# Identificar a composição dos registros selecionados
# quanto à presença de título e mensagem.
#
# Justificativa:
#
# Compreender quais combinações de campos textuais
# deverão ser contempladas na construção dos
# documentos da base de conhecimento do RAG.
#
# Ação:
#
# Contabilizar avaliações com ambos os campos,
# somente título ou somente mensagem,
# sem modificar os DataFrames existentes.

# Contagem das combinações de conteúdo textual
ambos_validos = (titulo_valido & mensagem_valida).sum()

somente_titulo = (
    titulo_valido & ~mensagem_valida
).sum()

somente_mensagem = (
    ~titulo_valido & mensagem_valida
).sum()

# Organização dos resultados
df_composicao_textual = pd.DataFrame({
    "situacao": [
        "Título e mensagem",
        "Somente título",
        "Somente mensagem"
    ],
    "quantidade": [
        ambos_validos,
        somente_titulo,
        somente_mensagem
    ]
})

df_composicao_textual["percentual"] = (
    df_composicao_textual["quantidade"]
    / len(df_reviews_texto)
    * 100
).round(2)

display(df_composicao_textual)

# Validação da composição
print(
    "Total contabilizado:",
    df_composicao_textual["quantidade"].sum()
)

print(
    "Total selecionado:",
    len(df_reviews_texto)
)



### Composição dos campos textuais da base de conhecimento

Após selecionar as 42.687 avaliações com conteúdo textual efetivo, foi investigada a disponibilidade dos campos `review_comment_title` e `review_comment_message`.

**Resultados encontrados:**

- Título e mensagem válidos: 9.829 avaliações (23,03%).
- Somente título válido: 1.737 avaliações (4,07%).
- Somente mensagem válida: 31.121 avaliações (72,91%).

A soma das três categorias corresponde aos 42.687 registros selecionados.

### Critérios propostos para a representação textual

A construção dos documentos da base de conhecimento seguirá inicialmente os seguintes critérios:

1. Quando título e mensagem estiverem disponíveis, ambos serão preservados no conteúdo textual do documento.
2. Quando apenas o título estiver disponível, ele será utilizado como conteúdo textual.
3. Quando apenas a mensagem estiver disponível, ela será utilizada como conteúdo textual.
4. Os campos serão identificados de maneira explícita quando necessário, preservando a distinção entre título e comentário.
5. Não serão acrescentadas informações que não estejam presentes nos registros originais.

### Justificativa

A estratégia busca aproveitar todo o conteúdo textual efetivo identificado na análise exploratória, evitando a exclusão de avaliações que possuem apenas um dos campos preenchidos.

A identificação dos campos poderá facilitar a interpretação do conteúdo durante a recuperação semântica e a geração de respostas fundamentadas.

### Relação com o RAG

Cada avaliação selecionada será considerada inicialmente como candidata à formação de um documento textual.

A definição final da estrutura dos documentos também deverá considerar os identificadores e metadados necessários à rastreabilidade das respostas.

**Decisão metodológica:** adotar a preservação de título e mensagem como princípio de preparação, sem realizar ainda a transformação definitiva dos registros.


In [0]:

# Objetivo:
#
# Investigar a unicidade dos identificadores
# das avaliações selecionadas para o RAG.
#
# Justificativa:
#
# A base vetorial deverá permitir identificar
# cada documento individualmente e recuperar
# sua origem sem ambiguidades.
#
# Ação:
#
# Contabilizar identificadores distintos,
# verificar repetições e avaliar a unicidade
# da combinação review_id + order_id,
# sem alterar os registros selecionados.

# Quantidade total de avaliações selecionadas
total_avaliacoes = len(df_reviews_texto)

# Quantidade de identificadores distintos
total_review_ids = (
    df_reviews_texto["review_id"].nunique()
)

total_order_ids = (
    df_reviews_texto["order_id"].nunique()
)

# Quantidade de registros além da primeira ocorrência
# de cada identificador
repeticoes_review_id = (
    df_reviews_texto["review_id"].duplicated().sum()
)

repeticoes_order_id = (
    df_reviews_texto["order_id"].duplicated().sum()
)

# Verificação da combinação dos dois identificadores
repeticoes_combinacao = (
    df_reviews_texto.duplicated(
        subset=["review_id", "order_id"]
    ).sum()
)

# Apresentação dos resultados
print("Total de avaliações:", total_avaliacoes)

print(
    "review_id distintos:",
    total_review_ids
)

print(
    "order_id distintos:",
    total_order_ids
)

print(
    "Ocorrências adicionais de review_id repetidos:",
    repeticoes_review_id
)

print(
    "Ocorrências adicionais de order_id repetidos:",
    repeticoes_order_id
)

print(
    "Ocorrências adicionais da combinação "
    "review_id + order_id:",
    repeticoes_combinacao
)

# Validação do índice original
print(
    "Índice do DataFrame é único:",
    df_reviews_texto.index.is_unique
)



### Identificação e rastreabilidade dos documentos

A investigação dos 42.687 registros selecionados identificou:

- 42.370 valores distintos de `review_id`;
- 42.540 valores distintos de `order_id`;
- 317 ocorrências adicionais de `review_id` repetidos;
- 147 ocorrências adicionais de `order_id` repetidos;
- nenhuma repetição da combinação `review_id` + `order_id`;
- índice único no DataFrame de trabalho.

### Interpretação

Os campos `review_id` e `order_id` não são individualmente exclusivos no subconjunto textual.

Entretanto, a combinação desses campos apresentou unicidade entre todos os registros selecionados.

O índice do DataFrame também é único, mas sua estabilidade depende da preservação da organização e da ordem dos registros de origem.

### Estratégia proposta

A identificação dos documentos será baseada inicialmente na combinação de `review_id` e `order_id`.

Os dois campos originais serão preservados separadamente como metadados, permitindo relacionar os documentos aos registros de origem.

A estrutura final deverá contemplar:

- identificador técnico único do documento;
- `review_id` original;
- `order_id` original;
- `review_score`;
- data de criação da avaliação;
- conteúdo textual preparado.

Outros metadados poderão ser incorporados conforme os requisitos de recuperação e filtragem do sistema RAG.

### Cuidados de implementação

A combinação dos identificadores deverá utilizar uma representação inequívoca, evitando ambiguidades na construção do identificador técnico.

A unicidade deverá ser validada novamente após a criação dos documentos e antes de sua inserção no armazenamento vetorial.

Nenhuma avaliação será excluída exclusivamente pela repetição isolada de `review_id` ou `order_id`.

### Relação com o Tech Challenge

A identificação única e a preservação dos metadados de origem contribuirão para a rastreabilidade das evidências utilizadas nas respostas do sistema RAG.

**Decisão metodológica:** adotar a combinação `review_id` + `order_id` como base inicial para os identificadores técnicos, mantendo ambos os campos originais e validando a unicidade na implementação.


In [0]:

# Objetivo:
#
# Inspecionar exemplos de títulos e mensagens
# antes de definir as regras de limpeza textual.
#
# Justificativa:
#
# A preparação dos documentos do RAG deve corrigir
# problemas de formatação sem alterar indevidamente
# o conteúdo escrito pelos clientes.
#
# Ação:
#
# Selecionar exemplos de avaliações com quebras
# de linha ou espaços nas extremidades e exibir
# suas representações textuais originais,
# sem modificar os DataFrames.

# Campos textuais a serem examinados
colunas_textuais = [
    "review_comment_title",
    "review_comment_message"
]

# Inspeção separada de cada campo
for coluna in colunas_textuais:

    # Consideração apenas de textos não nulos
    textos = df_reviews_texto[coluna].dropna()

    # Identificação de possíveis problemas de formatação
    possui_quebra = textos.str.contains(
        r"[\r\n]",
        regex=True
    )

    possui_espacos_extremidades = (
        textos.ne(textos.str.strip())
    )

    # Seleção dos exemplos
    exemplos = textos.loc[
        possui_quebra | possui_espacos_extremidades
    ].head(5)

    print(f"\nCampo: {coluna}")

    print(
        "Registros com quebras de linha:",
        possui_quebra.sum()
    )

    print(
        "Registros com espaços nas extremidades:",
        possui_espacos_extremidades.sum()
    )

    print("\nExemplos originais:")

    for indice, texto in exemplos.items():
        print(
            f"Índice {indice}: {repr(texto)}"
        )



### Definição das regras de limpeza textual

A inspeção dos registros selecionados identificou características de formatação nos campos textuais:

**Títulos**
- 1.997 registros com espaços em branco nas extremidades.
- Nenhum registro com quebras de linha.

**Mensagens**
- 9.433 registros com espaços em branco nas extremidades.
- 3.842 registros com quebras de linha.

As contagens representam características que podem ocorrer simultaneamente no mesmo registro.

### Estratégia de limpeza

A preparação textual adotará uma abordagem conservadora, priorizando a preservação do significado dos comentários.

Serão aplicadas as seguintes operações:

1. Substituir sequências de caracteres de espaço em branco, incluindo quebras de linha e tabulações, por um único espaço.
2. Remover espaços em branco no início e no final dos textos.
3. Tratar valores ausentes de maneira explícita durante a construção do conteúdo textual.

### Informações que serão preservadas

Não serão removidos ou alterados automaticamente:

- Acentos e caracteres próprios da língua portuguesa.
- Letras maiúsculas e minúsculas.
- Pontuação.
- Emojis e símbolos.
- Palavras repetidas.
- Erros ortográficos presentes nos comentários.
- Expressões curtas ou genéricas.

Também não serão aplicadas, nesta etapa, técnicas de remoção de stopwords, stemming ou lemmatização.

### Justificativa

O objetivo da limpeza é melhorar a consistência da representação textual utilizada na futura geração de embeddings, sem reescrever as manifestações dos clientes.

Transformações linguísticas mais agressivas poderiam eliminar informações relevantes para a recuperação semântica.

### Preservação da origem

Os campos textuais originais permanecerão disponíveis no DataFrame de origem.

As versões preparadas serão construídas separadamente, permitindo comparar os textos antes e depois da limpeza.

### Relação com o Tech Challenge

A estratégia contribui para a preparação de uma base de conhecimento textual adequada ao sistema RAG, preservando a fidelidade às avaliações dos clientes.

**Decisão metodológica:** aplicar somente normalização de espaços em branco nesta etapa, mantendo os comentários originais disponíveis para rastreabilidade.


In [0]:

# Objetivo:
#
# Implementar e testar uma função de limpeza textual
# que preserve o conteúdo das avaliações.
#
# Justificativa:
#
# Os documentos utilizados pelo RAG precisam ter
# formatação consistente, mas sem alterações
# desnecessárias no significado dos comentários.
#
# Ação:
#
# Criar uma função para normalizar espaços em branco,
# tratar valores ausentes e testar seu comportamento
# com exemplos conhecidos, sem alterar os DataFrames.

import re

def limpar_texto(texto):
    """
    Normaliza espaços em branco de um texto,
    preservando palavras, acentos e pontuação.

    Retorna uma string vazia para valores ausentes.
    """

    # Tratamento de valores ausentes
    if pd.isna(texto):
        return ""

    # Conversão para string
    texto = str(texto)

    # Substituição de sequências de espaços em branco
    texto = re.sub(r"\s+", " ", texto)

    # Remoção de espaços nas extremidades
    texto = texto.strip()

    return texto


# Exemplos para validação
exemplos_teste = [
    "  Ótimo Produto!  ",
    "Entrega rápida\r\nProduto excelente",
    "Muito   bom!",
    "Não chegou meu produto ",
    "Produto excelente! 👍",
    None,
    "   "
]

# Aplicação da função somente aos exemplos
for exemplo in exemplos_teste:

    resultado = limpar_texto(exemplo)

    print("Original:", repr(exemplo))
    print("Limpo:   ", repr(resultado))
    print("-" * 50)


In [0]:

# Objetivo:
#
# Aplicar a limpeza textual aos títulos e mensagens
# das avaliações selecionadas para o RAG.
#
# Justificativa:
#
# Preparar os textos para a construção dos documentos,
# mantendo os campos originais disponíveis para
# conferência e rastreabilidade.
#
# Ação:
#
# Criar um novo DataFrame de preparação, adicionar
# colunas com os textos normalizados e verificar
# a quantidade de registros e campos válidos.

# Criação de uma cópia independente para preparação
df_reviews_preparado = df_reviews_texto.copy()

# Limpeza dos títulos
df_reviews_preparado["titulo_limpo"] = (
    df_reviews_preparado["review_comment_title"]
    .map(limpar_texto)
)

# Limpeza das mensagens
df_reviews_preparado["mensagem_limpa"] = (
    df_reviews_preparado["review_comment_message"]
    .map(limpar_texto)
)

# Validação da quantidade de registros
print(
    "Avaliações selecionadas:",
    len(df_reviews_texto)
)

print(
    "Avaliações após a limpeza:",
    len(df_reviews_preparado)
)

# Validação dos campos textuais
print(
    "Títulos limpos não vazios:",
    df_reviews_preparado["titulo_limpo"].ne("").sum()
)

print(
    "Mensagens limpas não vazias:",
    df_reviews_preparado["mensagem_limpa"].ne("").sum()
)

# Verificação de avaliações com pelo menos um texto
possui_texto_limpo = (
    df_reviews_preparado["titulo_limpo"].ne("")
    | df_reviews_preparado["mensagem_limpa"].ne("")
)

print(
    "Avaliações com texto após a limpeza:",
    possui_texto_limpo.sum()
)

# Visualização de uma amostra
display(
    df_reviews_preparado[
        [
            "review_comment_title",
            "titulo_limpo",
            "review_comment_message",
            "mensagem_limpa"
        ]
    ].head(10)
)


In [0]:

# Objetivo:
#
# Construir o conteúdo textual de cada avaliação
# para a futura base de conhecimento do RAG.
#
# Justificativa:
#
# Os embeddings serão gerados a partir de uma
# representação textual consistente, que preserve
# título e mensagem e identifique sua origem.
#
# Ação:
#
# Criar uma função para compor o texto dos documentos,
# aplicá-la ao DataFrame preparado e validar
# a quantidade de conteúdos gerados.

def construir_texto_documento(titulo, mensagem):
    """
    Combina título e mensagem em uma representação
    textual estruturada, sem acrescentar informações
    ausentes na avaliação original.
    """

    partes = []

    # Inclusão do título, quando disponível
    if titulo:
        partes.append(f"Título: {titulo}")

    # Inclusão da mensagem, quando disponível
    if mensagem:
        partes.append(f"Comentário: {mensagem}")

    # Separação dos campos por quebra de linha
    return "\n".join(partes)


# Construção do conteúdo textual dos documentos
df_reviews_preparado["texto_documento"] = [
    construir_texto_documento(titulo, mensagem)
    for titulo, mensagem in zip(
        df_reviews_preparado["titulo_limpo"],
        df_reviews_preparado["mensagem_limpa"]
    )
]

# Validação dos conteúdos gerados
print(
    "Total de avaliações preparadas:",
    len(df_reviews_preparado)
)

print(
    "Documentos com conteúdo textual:",
    df_reviews_preparado["texto_documento"].ne("").sum()
)

print(
    "Documentos com conteúdo vazio:",
    df_reviews_preparado["texto_documento"].eq("").sum()
)

# Visualização de exemplos reais
print("\nExemplos de documentos preparados:")

for indice in [3, 9, 19]:
    print(f"\nÍndice original: {indice}")
    print(repr(
        df_reviews_preparado.loc[indice, "texto_documento"]
    ))
    print("-" * 50)


In [0]:

# Objetivo:
#
# Criar um identificador técnico único para cada
# documento textual da base de conhecimento.
#
# Justificativa:
#
# Os campos review_id e order_id apresentam
# repetições individuais. A combinação dos dois
# foi identificada como única no conjunto
# selecionado para o RAG.
#
# Ação:
#
# Construir um identificador a partir do par
# review_id e order_id e validar sua unicidade,
# sem modificar os identificadores originais.

import json

# Verificação de valores ausentes nos identificadores
print(
    "review_id ausentes:",
    df_reviews_preparado["review_id"].isna().sum()
)

print(
    "order_id ausentes:",
    df_reviews_preparado["order_id"].isna().sum()
)

# Verificação da unicidade da combinação original
duplicatas_combinacao = df_reviews_preparado.duplicated(
    subset=["review_id", "order_id"]
).sum()

print(
    "Combinações repetidas de review_id e order_id:",
    duplicatas_combinacao
)

# Criação do identificador técnico
df_reviews_preparado["documento_id"] = [
    json.dumps(
        [review_id, order_id],
        ensure_ascii=False,
        separators=(",", ":")
    )
    for review_id, order_id in zip(
        df_reviews_preparado["review_id"],
        df_reviews_preparado["order_id"]
    )
]

# Validação dos identificadores criados
print(
    "Total de documentos:",
    len(df_reviews_preparado)
)

print(
    "Identificadores técnicos distintos:",
    df_reviews_preparado["documento_id"].nunique()
)

print(
    "Identificadores técnicos duplicados:",
    df_reviews_preparado["documento_id"].duplicated().sum()
)

# Visualização de alguns identificadores
display(
    df_reviews_preparado[
        ["review_id", "order_id", "documento_id"]
    ].head(5)
)


In [0]:

# Objetivo:
#
# Organizar os documentos textuais e seus metadados
# em uma estrutura dedicada à base de conhecimento.
#
# Justificativa:
#
# O sistema RAG precisa associar cada texto recuperado
# aos registros originais, além de preservar atributos
# úteis para rastreabilidade e possíveis filtros.
#
# Ação:
#
# Selecionar os campos necessários, criar um DataFrame
# independente e validar sua integridade.

# Definição dos campos da base de conhecimento
colunas_documentos = [
    "documento_id",
    "review_id",
    "order_id",
    "review_score",
    "review_creation_date",
    "texto_documento"
]

# Criação da estrutura dedicada aos documentos
df_documentos_rag = (
    df_reviews_preparado[colunas_documentos]
    .copy()
)

# Validações de integridade
print(
    "Total de documentos:",
    len(df_documentos_rag)
)

print(
    "Identificadores únicos:",
    df_documentos_rag["documento_id"].nunique()
)

print(
    "Documentos com texto vazio:",
    df_documentos_rag["texto_documento"].eq("").sum()
)

print(
    "\nValores ausentes por campo:"
)

print(
    df_documentos_rag.isna().sum()
)

print(
    "\nTipos de dados:"
)

print(
    df_documentos_rag.dtypes
)

# Visualização da estrutura
display(
    df_documentos_rag.head(5)
)

In [0]:

# Objetivo:
#
# Validar e converter as datas de criação das
# avaliações para um tipo temporal adequado.
#
# Justificativa:
#
# Datas válidas permitem filtros temporais
# confiáveis na recuperação dos documentos
# e preservam a rastreabilidade das avaliações.
#
# Ação:
#
# Criar uma nova coluna de data convertida,
# verificar valores inválidos e inspecionar
# o intervalo temporal da base de conhecimento.

# Conversão sem alterar o campo original
df_documentos_rag["data_avaliacao"] = pd.to_datetime(
    df_documentos_rag["review_creation_date"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

# Validação da conversão
print(
    "Total de documentos:",
    len(df_documentos_rag)
)

print(
    "Datas convertidas com sucesso:",
    df_documentos_rag["data_avaliacao"].notna().sum()
)

print(
    "Datas inválidas ou ausentes:",
    df_documentos_rag["data_avaliacao"].isna().sum()
)

# Intervalo temporal
print(
    "\nData mais antiga:",
    df_documentos_rag["data_avaliacao"].min()
)

print(
    "Data mais recente:",
    df_documentos_rag["data_avaliacao"].max()
)

# Tipos dos campos temporais
print("\nTipos de dados:")

print(
    df_documentos_rag[
        ["review_creation_date", "data_avaliacao"]
    ].dtypes
)

# Comparação entre representação original e convertida
display(
    df_documentos_rag[
        ["review_creation_date", "data_avaliacao"]
    ].head(5)
)


In [0]:

# Objetivo:
#
# Validar as notas das avaliações presentes
# na base de documentos do RAG.
#
# Justificativa:
#
# O campo review_score poderá ser utilizado
# como metadado de filtragem na recuperação
# semântica. Seus valores precisam respeitar
# a escala original de 1 a 5.
#
# Ação:
#
# Verificar valores ausentes, notas fora da
# escala esperada e a distribuição das notas,
# sem alterar o DataFrame.

# Verificação de valores ausentes
notas_ausentes = (
    df_documentos_rag["review_score"]
    .isna()
    .sum()
)

# Identificação de notas fora da escala de 1 a 5
notas_invalidas = (
    ~df_documentos_rag["review_score"]
    .between(1, 5)
).sum()

# Distribuição das notas
distribuicao_notas = (
    df_documentos_rag["review_score"]
    .value_counts()
    .sort_index()
)

# Exibição dos resultados
print(
    "Total de documentos:",
    len(df_documentos_rag)
)

print(
    "Notas ausentes:",
    notas_ausentes
)

print(
    "Notas fora da escala de 1 a 5:",
    notas_invalidas
)

print("\nDistribuição das notas:")

print(distribuicao_notas)

print(
    "\nTotal da distribuição:",
    distribuicao_notas.sum()
)


In [0]:
# Objetivo: persistir documentos válidos no Volume sem modificar a camada raw.
# Justificativa: preservar a rastreabilidade e detectar alterações na base preparada.
# Ação: validar os documentos, comparar o JSONL existente e salvar a base.
from src.artefatos import COLUNAS, hash_documentos, sha256
DIRETORIO_PROCESSADOS = CFG["processed"]
colunas_exportacao = COLUNAS
base_exportacao = df_documentos_rag[colunas_exportacao].reset_index(drop=True)
validar_documentos(base_exportacao)
if ARQUIVO_DOCUMENTOS.is_file():
    existente = carregar_documentos(ARQUIVO_DOCUMENTOS)
    if hash_documentos(existente) != hash_documentos(base_exportacao):
        raise ValueError("JSONL existente difere da reconstrução. Preserve-o em outra pasta antes de gerar a nova base.")
    print("JSONL existente corresponde integralmente à reconstrução e foi preservado.")
else:
    base_exportacao.to_json(ARQUIVO_DOCUMENTOS, orient="records", lines=True,
                            force_ascii=False, date_format="iso")
print("Arquivo:", ARQUIVO_DOCUMENTOS)
print("Documentos:", len(base_exportacao))
print("SHA-256:", sha256(ARQUIVO_DOCUMENTOS))


In [0]:

# Objetivo:
#
# Validar a leitura da base de conhecimento
# persistida no formato JSON Lines.
#
# Justificativa:
#
# O pipeline de embeddings deverá conseguir
# recuperar os documentos diretamente do arquivo
# processado, sem depender da execução das
# etapas anteriores do notebook.
#
# Ação:
#
# Ler o arquivo JSONL em um novo DataFrame
# e comparar sua estrutura e conteúdo com
# os documentos preparados originalmente.

# Leitura independente do arquivo exportado
df_documentos_recarregado = pd.read_json(
    ARQUIVO_DOCUMENTOS,
    orient="records",
    lines=True,
    dtype={
        "documento_id": "str",
        "review_id": "str",
        "order_id": "str",
        "review_score": "int64",
        "review_creation_date": "str",
        "texto_documento": "str"
    },
    convert_dates=False
)

# Validação da quantidade de registros
print(
    "Documentos originais:",
    len(df_documentos_rag)
)

print(
    "Documentos recarregados:",
    len(df_documentos_recarregado)
)

# Validação da estrutura
print(
    "\nColunas preservadas:",
    df_documentos_recarregado.columns.tolist()
    == colunas_exportacao
)

# Validação dos identificadores
print(
    "Identificadores únicos:",
    df_documentos_recarregado["documento_id"].nunique()
)

# Verificação de textos vazios
print(
    "Textos vazios:",
    df_documentos_recarregado["texto_documento"]
    .eq("")
    .sum()
)

# Comparação exata com os campos exportados
conteudo_preservado = (
    df_documentos_recarregado.reset_index(drop=True)
    .equals(
        df_documentos_rag[
            colunas_exportacao
        ].reset_index(drop=True)
    )
)

print(
    "\nConteúdo preservado integralmente:",
    conteudo_preservado
)

# Visualização dos tipos recuperados
print("\nTipos de dados após a leitura:")

print(
    df_documentos_recarregado.dtypes
)

assert conteudo_preservado, "JSONL recarregado difere da base exportada."
validar_documentos(df_documentos_recarregado)



## Encerramento — Preparação da Base de Conhecimento

### Objetivo alcançado

Este notebook realizou a seleção, a limpeza, a estruturação e a persistência dos comentários de clientes da Olist que serão utilizados como base de conhecimento do sistema RAG.

### Resultados finais

| Indicador | Resultado |
|---|---:|
| Avaliações na base original | 99.224 |
| Avaliações com conteúdo textual | 42.687 |
| Documentos preparados | 42.687 |
| Identificadores técnicos únicos | 42.687 |
| Documentos com texto vazio | 0 |
| Notas inválidas ou ausentes | 0 |
| Datas inválidas ou ausentes | 0 |
| Linhas exportadas para JSONL | 42.687 |
| Tamanho do arquivo exportado | 14,51 MB |

### Decisões metodológicas

**Seleção dos documentos**

Foram selecionadas as avaliações que possuíam conteúdo efetivo no título, na mensagem ou em ambos os campos.

**Limpeza textual**

Foi aplicada uma normalização conservadora dos espaços em branco, preservando palavras, acentos, pontuação, emojis e a escrita original dos clientes.

**Composição do conteúdo**

Os documentos foram construídos com os rótulos `Título:` e `Comentário:`, utilizados somente quando o respectivo campo possuía conteúdo.

**Identificação e rastreabilidade**

Cada documento recebeu um identificador técnico baseado na combinação de `review_id` e `order_id`, cuja unicidade foi validada no conjunto selecionado.

Os identificadores originais foram preservados.

**Metadados**

Foram mantidos os identificadores de origem, a nota da avaliação e a data de criação, permitindo rastreabilidade e futuros filtros.

**Persistência**

A base foi exportada em formato JSON Lines, com codificação UTF-8, e posteriormente recarregada para validação.

A comparação entre os campos exportados e os documentos recarregados confirmou a preservação integral dos valores, tipos e ordem dos registros.

### Arquivo produzido

`data/processed/olist_reviews_documentos_rag.jsonl`

Esse arquivo constitui a base textual preparada para as próximas etapas do projeto.

### Limitações e cuidados

A base contém somente avaliações com conteúdo textual efetivo, não representando necessariamente a distribuição de satisfação de todos os clientes.

A limpeza textual não incluiu correção ortográfica nem interpretação de sentimentos.

A preparação dos documentos não equivale à implementação completa do sistema RAG: ainda serão necessários embeddings, armazenamento vetorial, recuperação semântica e geração de respostas fundamentadas.

### Próxima etapa

O próximo notebook será dedicado à geração de embeddings e à implementação da busca vetorial.

Os documentos preparados serão utilizados como entrada para a indexação semântica, preservando seus identificadores e metadados.

### Conclusão

A etapa de preparação da base de conhecimento foi concluída e validada.

O projeto dispõe agora de 42.687 documentos textuais estruturados, identificáveis e persistidos, prontos para alimentar a etapa de indexação do sistema RAG.


In [0]:
# Objetivo: registrar a integridade dos documentos preparados.
# Justificativa: fornecer evidência da execução Databricks para a entrega.
# Ação: salvar indicadores e hash da sequência dos documentos.
import json
relatorio_02 = {"documentos": len(df_documentos_recarregado),
                "hash_documentos_ordenados": hash_documentos(df_documentos_recarregado),
                "conteudo_preservado": bool(conteudo_preservado),
                "jsonl_sha256": sha256(ARQUIVO_DOCUMENTOS)}
(CFG["reports"] / "02_preparacao.json").write_text(
    json.dumps(relatorio_02, ensure_ascii=False, indent=2), encoding="utf-8")
print(relatorio_02)
